# LOB Data Extraction and Cleaning Pipeline

## Dataset Sufficiency Check

Based on the dataset description and local files, there is enough information to build extraction and cleaning for model training:
- File naming provides split, auction flag, normalization type, and fold.
- Each file provides 149 rows where rows 1-144 are features and rows 145-149 are labels.
- Label values are expected in {1, 2, 3}.

## Notebook Sections
This notebook is organized into coherent sections:
1. Configuration, imports, and paper-based feature schema (`u1`-`u9`)
2. File discovery and metadata parsing
3. Raw file extraction into a tabular DataFrame
4. Data cleaning for direct model training
5. Final train/test model-ready matrices
6. Feature schema sanity checks
7. PyTorch logistic regression baseline

## 1) Setting Up

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("Data")
EXPECTED_FEATURE_ROWS = 144
EXPECTED_LABEL_ROWS = 5
EXPECTED_TOTAL_ROWS = EXPECTED_FEATURE_ROWS + EXPECTED_LABEL_ROWS


def build_paper_feature_names(n_levels: int = 10) -> list[str]:
    names: list[str] = []

    for i in range(1, n_levels + 1):
        names.extend([
            f"u1_ask_price_l{i}",
            f"u1_ask_volume_l{i}",
            f"u1_bid_price_l{i}",
            f"u1_bid_volume_l{i}",
        ])

    for i in range(1, n_levels + 1):
        names.extend([f"u2_spread_l{i}", f"u2_midprice_l{i}"])

    names.extend(["u3_ask_price_range_l10_l1", "u3_bid_price_range_l1_l10"])
    for i in range(1, n_levels):
        names.append(f"u3_abs_ask_price_diff_l{i+1}_l{i}")
    for i in range(1, n_levels):
        names.append(f"u3_abs_bid_price_diff_l{i+1}_l{i}")

    names.extend([
        "u4_mean_ask_price",
        "u4_mean_bid_price",
        "u4_mean_ask_volume",
        "u4_mean_bid_volume",
    ])

    names.extend(["u5_sum_spread", "u5_sum_volume_imbalance"])

    for i in range(1, n_levels + 1):
        names.extend([
            f"u6_dask_price_dt_l{i}",
            f"u6_dbid_price_dt_l{i}",
            f"u6_dask_volume_dt_l{i}",
            f"u6_dbid_volume_dt_l{i}",
        ])

    event_types = [
        "trade",
        "order",
        "cancellation",
        "deletion",
        "exec_visible",
        "exec_hidden",
    ]
    names.extend([f"u7_intensity_{event}" for event in event_types])
    names.extend([f"u8_rel_intensity_gt_long_{event}" for event in event_types])
    names.extend([f"u9_dintensity_dt_{event}" for event in event_types])

    if len(names) != EXPECTED_FEATURE_ROWS:
        raise ValueError(
            f"Paper feature schema must have {EXPECTED_FEATURE_ROWS} features, got {len(names)}"
        )

    return names


FEATURE_COLS = build_paper_feature_names(n_levels=10)
LABEL_COLS = [f"label_{i}" for i in range(1, EXPECTED_LABEL_ROWS + 1)]
META_COLS = ["file_name", "split", "auction", "normalization", "fold"]

print(f"Data directory exists: {DATA_DIR.exists()} -> {DATA_DIR.resolve()}")
print(f"Feature columns configured from paper schema: {len(FEATURE_COLS)}")

Data directory exists: True -> C:\Users\prane\Work\Coding\Projects\LOBForecasting\Data
Feature columns configured from paper schema: 144


## 2) File Discovery + Metadata Parsing

We parse file names of the form:
- `Train_Dst_NoAuction_DecPre_CF_7.txt`
- `Test_Dst_NoAuction_DecPre_CF_8.txt`

Parsed metadata fields:
- `split`: train/test
- `auction`: with_auction/no_auction
- `normalization`: zscore/minmax/decpre (or raw token if new)
- `fold`: cross-validation fold integer

In [2]:
FILE_PATTERN = re.compile(
    r"^(Train|Test)_Dst_(NoAuction|Auction)_([A-Za-z0-9]+)_CF_(\d+)\.txt$",
    re.IGNORECASE,
)


def parse_file_metadata(file_path: Path) -> dict:
    match = FILE_PATTERN.match(file_path.name)
    if not match:
        raise ValueError(
            f"Unexpected filename format: {file_path.name}. "
            "Expected e.g. Train_Dst_NoAuction_DecPre_CF_7.txt"
        )

    split_raw, auction_raw, norm_raw, fold_raw = match.groups()

    split = split_raw.lower()
    auction = "no_auction" if auction_raw.lower() == "noauction" else "with_auction"

    normalization_lookup = {
        "zscore": "zscore",
        "minmax": "minmax",
        "decpre": "decpre",
    }
    normalization = normalization_lookup.get(norm_raw.lower(), norm_raw.lower())

    return {
        "file_name": file_path.name,
        "split": split,
        "auction": auction,
        "normalization": normalization,
        "fold": int(fold_raw),
    }


def discover_data_files(data_dir: Path) -> list[Path]:
    files = sorted(data_dir.glob("*.txt"))
    if not files:
        raise FileNotFoundError(
            f"No .txt files found in {data_dir.resolve()}. "
            "Place Kaggle dataset files inside Data/."
        )
    return files


all_files = discover_data_files(DATA_DIR)
metadata_preview = pd.DataFrame([parse_file_metadata(path) for path in all_files])
metadata_preview

,file_name,split,auction,normalization,fold
0,Test_Dst_NoAuction_DecPre_CF_7.txt,test,no_auction,decpre,7
1,Test_Dst_NoAuction_DecPre_CF_8.txt,test,no_auction,decpre,8
2,Test_Dst_NoAuction_DecPre_CF_9.txt,test,no_auction,decpre,9
3,Train_Dst_NoAuction_DecPre_CF_7.txt,train,no_auction,decpre,7


## 3) Raw Extraction into a Tabular Dataframe

Important handling logic:
- Raw file has 149 rows and many columns.
- Rows correspond to variables; columns correspond to observations.
- We transpose so each output row is one observation.
- Feature columns are assigned directly using paper-based names (`u1` to `u9` decomposition).
- Label columns are `label_1`..`label_5`.

In [3]:
def read_lob_txt_to_dataframe(file_path: Path) -> pd.DataFrame:
    raw = pd.read_csv(file_path, sep=r"\s+", header=None, engine="c")

    if raw.shape[0] != EXPECTED_TOTAL_ROWS:
        raise ValueError(
            f"{file_path.name}: expected {EXPECTED_TOTAL_ROWS} rows, found {raw.shape[0]}."
        )

    transposed = raw.T.reset_index(drop=True)
    transposed.columns = FEATURE_COLS + LABEL_COLS

    metadata = parse_file_metadata(file_path)
    for key, value in metadata.items():
        transposed[key] = value

    cols_order = [
        "file_name",
        "split",
        "auction",
        "normalization",
        "fold",
        *FEATURE_COLS,
        *LABEL_COLS,
    ]
    return transposed[cols_order]


def build_raw_dataset(data_dir: Path) -> pd.DataFrame:
    file_paths = discover_data_files(data_dir)
    frames = [read_lob_txt_to_dataframe(path) for path in file_paths]
    combined = pd.concat(frames, ignore_index=True)
    return combined


raw_df = build_raw_dataset(DATA_DIR)
print("Raw combined shape:", raw_df.shape)
raw_df.head(3)

Raw combined shape: (394337, 154)


,file_name,split,auction,normalization,fold,u1_ask_price_l1,u1_ask_volume_l1,u1_bid_price_l1,u1_bid_volume_l1,u1_ask_price_l2,u1_ask_volume_l2,u1_bid_price_l2,u1_bid_volume_l2,u1_ask_price_l3,u1_ask_volume_l3,u1_bid_price_l3,u1_bid_volume_l3,u1_ask_price_l4,u1_ask_volume_l4,u1_bid_price_l4,u1_bid_volume_l4,u1_ask_price_l5,u1_ask_volume_l5,u1_bid_price_l5,u1_bid_volume_l5,u1_ask_price_l6,u1_ask_volume_l6,u1_bid_price_l6,u1_bid_volume_l6,u1_ask_price_l7,u1_ask_volume_l7,u1_bid_price_l7,u1_bid_volume_l7,u1_ask_price_l8,u1_ask_volume_l8,u1_bid_price_l8,u1_bid_volume_l8,u1_ask_price_l9,u1_ask_volume_l9,u1_bid_price_l9,u1_bid_volume_l9,u1_ask_price_l10,u1_ask_volume_l10,u1_bid_price_l10,u1_bid_volume_l10,u2_spread_l1,u2_midprice_l1,u2_spread_l2,u2_midprice_l2,u2_spread_l3,u2_midprice_l3,u2_spread_l4,u2_midprice_l4,u2_spread_l5,u2_midprice_l5,u2_spread_l6,u2_midprice_l6,u2_spread_l7,u2_midprice_l7,u2_spread_l8,u2_midprice_l8,u2_spread_l9,u2_midprice_l9,u2_spread_l10,u2_midprice_l10,u3_ask_price_range_l10_l1,u3_bid_price_range_l1_l10,u3_abs_ask_price_diff_l2_l1,u3_abs_ask_price_diff_l3_l2,u3_abs_ask_price_diff_l4_l3,u3_abs_ask_price_diff_l5_l4,u3_abs_ask_price_diff_l6_l5,u3_abs_ask_price_diff_l7_l6,u3_abs_ask_price_diff_l8_l7,u3_abs_ask_price_diff_l9_l8,u3_abs_ask_price_diff_l10_l9,u3_abs_bid_price_diff_l2_l1,u3_abs_bid_price_diff_l3_l2,u3_abs_bid_price_diff_l4_l3,u3_abs_bid_price_diff_l5_l4,u3_abs_bid_price_diff_l6_l5,u3_abs_bid_price_diff_l7_l6,u3_abs_bid_price_diff_l8_l7,u3_abs_bid_price_diff_l9_l8,u3_abs_bid_price_diff_l10_l9,u4_mean_ask_price,u4_mean_bid_price,u4_mean_ask_volume,u4_mean_bid_volume,u5_sum_spread,u5_sum_volume_imbalance,u6_dask_price_dt_l1,u6_dbid_price_dt_l1,u6_dask_volume_dt_l1,u6_dbid_volume_dt_l1,u6_dask_price_dt_l2,u6_dbid_price_dt_l2,u6_dask_volume_dt_l2,u6_dbid_volume_dt_l2,u6_dask_price_dt_l3,u6_dbid_price_dt_l3,u6_dask_volume_dt_l3,u6_dbid_volume_dt_l3,u6_dask_price_dt_l4,u6_dbid_price_dt_l4,u6_dask_volume_dt_l4,u6_dbid_volume_dt_l4,u6_dask_price_dt_l5,u6_dbid_price_dt_l5,u6_dask_volume_dt_l5,u6_dbid_volume_dt_l5,u6_dask_price_dt_l6,u6_dbid_price_dt_l6,u6_dask_volume_dt_l6,u6_dbid_volume_dt_l6,u6_dask_price_dt_l7,u6_dbid_price_dt_l7,u6_dask_volume_dt_l7,u6_dbid_volume_dt_l7,u6_dask_price_dt_l8,u6_dbid_price_dt_l8,u6_dask_volume_dt_l8,u6_dbid_volume_dt_l8,u6_dask_price_dt_l9,u6_dbid_price_dt_l9,u6_dask_volume_dt_l9,u6_dbid_volume_dt_l9,u6_dask_price_dt_l10,u6_dbid_price_dt_l10,u6_dask_volume_dt_l10,u6_dbid_volume_dt_l10,u7_intensity_trade,u7_intensity_order,u7_intensity_cancellation,u7_intensity_deletion,u7_intensity_exec_visible,u7_intensity_exec_hidden,u8_rel_intensity_gt_long_trade,u8_rel_intensity_gt_long_order,u8_rel_intensity_gt_long_cancellation,u8_rel_intensity_gt_long_deletion,u8_rel_intensity_gt_long_exec_visible,u8_rel_intensity_gt_long_exec_hidden,u9_dintensity_dt_trade,u9_dintensity_dt_order,u9_dintensity_dt_cancellation,u9_dintensity_dt_deletion,u9_dintensity_dt_exec_visible,u9_dintensity_dt_exec_hidden,label_1,label_2,label_3,label_4,label_5
0,Test_Dst_NoAuction_DecPre_CF_7.txt,test,no_auction,decpre,7,0.2666,0.00129,0.2654,0.00225,0.2669,0.00246,0.2653,0.01033,0.2670,0.00050,0.2651,0.00289,0.2671,0.00143,0.2650,0.02000,0.2674,0.00180,0.2648,0.00121,0.2677,0.01000,0.2647,0.00425,0.2681,0.00021,0.2646,0.00263,0.2688,0.00500,0.2644,0.00169,0.2690,0.0124,0.2641,0.00282,0.2700,0.00200,0.2638,0.00156,0.12,0.16,0.19,0.021,0.026,0.030,0.035,0.044,0.049,0.062,0.26600,0.26610,0.26605,0.26605,0.26610,0.26620,0.26635,0.26660,0.26655,0.26690,0.03,0.01,0.01,0.03,0.03,0.04,0.07,0.02,0.10,0.034,0.01,0.02,0.01,0.02,0.01,0.01,0.02,0.03,0.03,0.028,0.26786,0.26472,0.03709,0.04963,0.0314,0.266290,0.000038,0.000003,0.011111,-0.00045,0.000013,-0.000003,0.005556,-0.004489,0.000013,0.000001,0.011111,0.002309,0.000779,0.000203,0.005556,-0.007211,0.001026,-0.000231,0.013889,0.010275,0.001780,0.000106,-0.002778,0.000224,0.000584,0.000233,-0.005556,-0.000261,0.001754,0.000069,-0.008333,0.000314,0.000830,-0.000313,-0.008

## 4) Cleaning for Direct Model Training

Cleaning steps applied:
- Convert all feature and label columns to numeric.
- Replace `inf/-inf` with `NaN`.
- Impute feature `NaN` values using median (robust default).
- Remove rows with invalid label values (must be in {1, 2, 3}).
- Drop exact duplicate rows.
- Add encoded labels (`label_i_encoded`) with mapping {1:0, 2:1, 3:2}.

In [4]:
def clean_lob_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    cleaned[FEATURE_COLS + LABEL_COLS] = cleaned[FEATURE_COLS + LABEL_COLS].apply(
        pd.to_numeric, errors="coerce"
    )

    cleaned[FEATURE_COLS] = cleaned[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    cleaned[LABEL_COLS] = cleaned[LABEL_COLS].replace([np.inf, -np.inf], np.nan)

    feature_medians = cleaned[FEATURE_COLS].median()
    cleaned[FEATURE_COLS] = cleaned[FEATURE_COLS].fillna(feature_medians)

    valid_label_mask = np.ones(len(cleaned), dtype=bool)
    for col in LABEL_COLS:
        valid_label_mask &= cleaned[col].isin([1, 2, 3])
    cleaned = cleaned.loc[valid_label_mask].copy()

    cleaned = cleaned.drop_duplicates().reset_index(drop=True)

    label_map = {1: 0, 2: 1, 3: 2}
    for col in LABEL_COLS:
        cleaned[f"{col}_encoded"] = cleaned[col].map(label_map).astype("int8")

    return cleaned


clean_df = clean_lob_dataframe(raw_df)
print("Clean shape:", clean_df.shape)
print("Rows dropped during cleaning:", len(raw_df) - len(clean_df))
clean_df[META_COLS + LABEL_COLS + [f"label_{i}_encoded" for i in range(1, 6)]].head(3)

Clean shape: (391920, 159)
Rows dropped during cleaning: 2417


,file_name,split,auction,normalization,fold,label_1,label_2,label_3,label_4,label_5,label_1_encoded,label_2_encoded,label_3_encoded,label_4_encoded,label_5_encoded
0,Test_Dst_NoAuction_DecPre_CF_7.txt,test,no_auction,decpre,7,2.0,2.0,2.0,2.0,2.0,1,1,1,1,1
1,Test_Dst_NoAuction_DecPre_CF_7.txt,test,no_auction,decpre,7,2.0,2.0,2.0,2.0,2.0,1,1,1,1,1
2,Test_Dst_NoAuction_DecPre_CF_7.txt,test,no_auction,decpre,7,3.0,3.0,2.0,2.0,2.0,2,2,1,1,1


## 5) Build Model-Ready Train/Test Matrices

Use `make_model_inputs(...)` to select one of the five classification targets.
By default, `label_1` is used and encoded to 0/1/2.

In [5]:
def make_model_inputs(
    df: pd.DataFrame,
    target_label: str = "label_1",
) -> dict:
    if target_label not in LABEL_COLS:
        raise ValueError(f"target_label must be one of {LABEL_COLS}, got {target_label}")

    target_encoded = f"{target_label}_encoded"

    train_df = df[df["split"] == "train"].copy()
    test_df = df[df["split"] == "test"].copy()

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[target_encoded]
    X_test = test_df[FEATURE_COLS]
    y_test = test_df[target_encoded]

    return {
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "target_label": target_label,
    }


model_data = make_model_inputs(clean_df, target_label="label_1")

print("X_train:", model_data["X_train"].shape)
print("y_train:", model_data["y_train"].shape)
print("X_test:", model_data["X_test"].shape)
print("y_test:", model_data["y_test"].shape)

print("\nTarget distribution (train):")
print(model_data["y_train"].value_counts(normalize=True).sort_index())

print("\nTarget distribution (test):")
print(model_data["y_test"].value_counts(normalize=True).sort_index())

X_train: (252584, 144)
y_train: (252584,)
X_test: (139336, 144)
y_test: (139336,)

Target distribution (train):
label_1_encoded
0    0.198892
1    0.602128
2    0.198979
Name: proportion, dtype: float64

Target distribution (test):
label_1_encoded
0    0.151877
1    0.706164
2    0.141959
Name: proportion, dtype: float64


## 6) Feature Schema Sanity Check

This section validates that extraction is using the paper-based feature schema directly and shows a compact view of the configured names.

In [6]:
assert len(FEATURE_COLS) == EXPECTED_FEATURE_ROWS
assert all(col in raw_df.columns for col in FEATURE_COLS)

feature_map_df = pd.DataFrame(
    {
        "feature_index": range(1, EXPECTED_FEATURE_ROWS + 1),
        "paper_feature_name": FEATURE_COLS,
    }
)

print("Feature schema check passed.")
print("First 12 feature names:")
display(feature_map_df.head(12))

print("Last 12 feature names:")
display(feature_map_df.tail(12))

Feature schema check passed.
First 12 feature names:


,feature_index,paper_feature_name
0,1,u1_ask_price_l1
1,2,u1_ask_volume_l1
2,3,u1_bid_price_l1
3,4,u1_bid_volume_l1
4,5,u1_ask_price_l2
5,6,u1_ask_volume_l2
6,7,u1_bid_price_l2
7,8,u1_bid_volume_l2
8,9,u1_ask_price_l3
9,10,u1_ask_volume_l3


Last 12 feature names:


,feature_index,paper_feature_name
132,133,u8_rel_intensity_gt_long_trade
133,134,u8_rel_intensity_gt_long_order
134,135,u8_rel_intensity_gt_long_cancellation
135,136,u8_rel_intensity_gt_long_deletion
136,137,u8_rel_intensity_gt_long_exec_visible
137,138,u8_rel_intensity_gt_long_exec_hidden
138,139,u9_dintensity_dt_trade
139,140,u9_dintensity_dt_order
140,141,u9_dintensity_dt_cancellation
141,142,u9_dintensity_dt_deletion


## 7) PyTorch Logistic Regression Baseline (Very Short Horizon)

Target used:
- `label_1` (the shortest horizon in this dataset setup)

Important interpretation:
- The dataset provides class labels for thresholded mid-price change, not raw return values.
- So this baseline predicts **direction + movement bucket** (`down`, `stationary`, `up`) rather than exact continuous magnitude.

Efficiency choices included:
- Vectorized preprocessing with `float32`
- Train/validation split with stratification
- `DataLoader` mini-batching
- GPU auto-detection, optional AMP for CUDA
- Simple linear head (`nn.Linear`) for true logistic regression

In [7]:
import time
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

TARGET_LABEL = "label_1"
TARGET_ENCODED = f"{TARGET_LABEL}_encoded"

train_full = clean_df[clean_df["split"] == "train"].copy()
test_full = clean_df[clean_df["split"] == "test"].copy()

X = train_full[FEATURE_COLS].to_numpy(dtype=np.float32)
y = train_full[TARGET_ENCODED].to_numpy(dtype=np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_test = test_full[FEATURE_COLS].to_numpy(dtype=np.float32)
y_test = test_full[TARGET_ENCODED].to_numpy(dtype=np.int64)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

batch_size = 4096
num_workers = 0
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)


class LogisticRegressionTorch(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.linear(features)


model = LogisticRegressionTorch(input_dim=len(FEATURE_COLS), num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)

amp_enabled = device.type == "cuda"
scaler_amp = torch.amp.GradScaler("cuda", enabled=amp_enabled)


def evaluate_accuracy(model: nn.Module, data_loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            logits = model(batch_x)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
    return correct / max(total, 1)


def train_model(model: nn.Module, epochs: int = 8) -> dict:
    best_state = None
    best_val_acc = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        total_samples = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(batch_x)
                loss = criterion(logits, batch_y)

            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()

            batch_size_local = batch_y.size(0)
            epoch_loss += loss.item() * batch_size_local
            total_samples += batch_size_local

        train_loss = epoch_loss / max(total_samples, 1)
        val_acc = evaluate_accuracy(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return {"best_val_acc": best_val_acc, "history": history}


start = time.perf_counter()
training_output = train_model(model, epochs=8)
train_seconds = time.perf_counter() - start

val_acc = evaluate_accuracy(model, val_loader)
test_acc = evaluate_accuracy(model, test_loader)

print("\nBest validation accuracy during training:", f"{training_output['best_val_acc']:.4f}")
print("Validation accuracy (best checkpoint):", f"{val_acc:.4f}")
print("Test accuracy:", f"{test_acc:.4f}")
print("Training time (seconds):", f"{train_seconds:.1f}")

Device: cpu
Epoch 01 | train_loss=0.9795 | val_acc=0.6033
Epoch 02 | train_loss=0.9079 | val_acc=0.6080
Epoch 03 | train_loss=0.9056 | val_acc=0.6087
Epoch 04 | train_loss=0.9053 | val_acc=0.6075
Epoch 05 | train_loss=0.9061 | val_acc=0.6083
Epoch 06 | train_loss=0.9059 | val_acc=0.6052
Epoch 07 | train_loss=0.9057 | val_acc=0.6073
Epoch 08 | train_loss=0.9060 | val_acc=0.6063

Best validation accuracy during training: 0.6087
Validation accuracy (best checkpoint): 0.6087
Test accuracy: 0.7077
Training time (seconds): 16.0
